# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(url)

# Access metadata (as a Python object, not as a dictionary)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Record sets and their fields are referenced by their `@id` as per Croissant best practices.**

In [ ]:
# List all record set `@id`s in the dataset
record_sets = dataset.metadata.record_sets
if not record_sets:
    raise RuntimeError("No record sets found in this dataset.")
print("Available record sets and their fields (by @id):\n")
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the main record set and field IDs for this dataset
# We'll select the first available record set for exploration

main_record_set_id = record_sets[0].id
print(f"Extracting from record set: {main_record_set_id}")

# You can repeat for multiple record sets if desired
all_record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in all_record_set_ids:
    print(f"Loading records for: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show fields (columns) for the main record set
print(f"Columns for main record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes removing outliers, transforming data distributions, or grouping data by key attributes.

**All field (column) references below are by their `@id` as recommended.**

> *Adjust field `@id`s and threshold as relevant to the dataset for meaningful analysis.*

In [ ]:
# --- Example: Filtering and Normalizing a Numeric Field ---

df = dataframes[main_record_set_id]
print("Sample columns:")
print(df.columns.tolist())

# Choose a numeric field to analyze. Update these as needed for the dataset fields you saw above.
# For example, let's use the UniProt MW (molecular weight) field if it is present

# Try to auto-detect a numeric field
numeric_field_id = None
for col in df.columns:
    # Heuristically pick a column with possible numeric data
    if "weight" in col.lower() or "mw" in col.lower() or "count" in col.lower() or "abundance" in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback to just using the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id is None:
    raise RuntimeError("No numeric field detected in the main record set. Please inspect dataset and set manually.")

print(f"Using numeric field: {numeric_field_id}")

# Filter out missing values to prevent errors below
df = df[df[numeric_field_id].notnull()]

# Choose a threshold for exploration (e.g., mean of the column)
threshold = df[numeric_field_id].mean()

# Filtering records above the threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by another field, e.g., 'accession' or 'description' if relevant
# Auto-detect a categorical/group field
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and ("desc" in col.lower() or "accession" in col.lower() or "id" in col.lower()):
        group_field_id = col
        break
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped average {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If there is a group field, make a boxplot
if group_field_id is not None:
    top_groups = df[group_field_id].value_counts().nlargest(10).index.tolist()
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
    plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset:

- Loaded and described the dataset and metadata using `mlcroissant`.
- Inspected record sets and their field `@id`s following Croissant best practices.
- Extracted data into pandas DataFrames by `@id` for flexible, programmatic access.
- Selected a key numeric field for filtering, normalization, and group analysis.
- Visualized field distributions and group differences.

The dataset provides a rich set of protein analysis results for downstream proteomic and bioinformatics research. Refer to the field and record set `@id`s for precise, reproducible workflows.